Training with cross-validation for the XGBoost base model. The XGBoost model predicts the target class (Drug) from the features (age, race/ethnicity, sex, family income, insurance coverage, prescription strength, prescription day supply, prescription quantity, and prescription form).

In [ ]:
# Install required libraries.
# xgboost 2.1+ has native categorical support and handles missing values via
# learned default directions — no imputation needed for structural NaNs.
# permetrics provides our evaluation metrics.
!pip install 'xgboost>=2.1.0' permetrics

ERROR: Invalid requirement: "'xgboost": Expected package name at the start of dependency specifier
    'xgboost
    ^


In [ ]:
# Load all required libraries.
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
import sklearn
print('XGBoost version:     ', xgb.__version__)
print('pandas version:      ', pd.__version__)
print('numpy version:       ', np.__version__)
print('scikit-learn version:', sklearn.__version__)

# Hard stop if XGBoost version is below 2.1.0.
# Native categorical support and learned missing directions
# require XGBoost 2.1+ — earlier versions will silently produce wrong results.
from packaging import version
assert version.parse(xgb.__version__) >= version.parse('2.1.0'), \
    f"ERROR: XGBoost 2.1+ required, found {xgb.__version__} — run: pip install 'xgboost>=2.1.0'"

print('\nAll version checks passed.')

XGBoost version:      3.2.0
pandas version:       2.2.3
numpy version:        2.2.3
scikit-learn version: 1.6.1

All version checks passed.


In [ ]:
# Load the super integrated dataset (2014-2021).
# The super dataset contains demographics + prescription features
# (Quantity, Form, Strength, Day_Supply) + Person_ID.

DATA_DIR   = './'  # change this to your data path if needed
SUPER_DATA = f'{DATA_DIR}super_integrated_data.csv'

super_df = pd.read_csv(
    SUPER_DATA,
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

# Drop auto-generated index column if present
if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

Super dataset shape: (904140, 14)

Super dataset columns: ['Observation_ID', 'Person_ID', 'Household_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Year', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64


In [ ]:
# Verify Person_ID is present in the dataset.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
assert 'Person_ID' in super_df.columns, \
    "ERROR: Person_ID not found in dataset — check super_integrated_data.csv."
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs — check super_integrated_data.csv."

print('Person_ID verified.')
print('Unique persons:', super_df['Person_ID'].nunique())

Person_ID verified.
Unique persons: 127415


In [ ]:
# EDA: confirm unique persons, drugs, distribution, missing values.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:  ', super_df['Drug'].nunique())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

print('\nMissing values per column:')
print(super_df.isnull().sum())

print('\nMissing value interpretation:')
print("  Quantity NaNs correspond to 'no prescriptions' rows — left as NaN for XGBoost native handling.")
print("  Strength / Day_Supply NaNs: real missing values — imputed inside each fold.")
print("  'no prescriptions' count:", (super_df['Drug'] == 'no prescriptions').sum())

# Check class imbalance ratio — max count / min count.
# Confirms severe imbalance and justifies reporting macro metrics
# which treat rare and common drugs equally — not just overall accuracy.
counts = super_df['Drug'].value_counts()
print(f'\nMost common drug count:  {counts.max():,}')
print(f'Rarest drug count:       {counts.min():,}')
print(f'Imbalance ratio:         {counts.max() / counts.min():.1f}x')

# Hard stop if drug count is not 217 (216 drugs + "no prescriptions").
assert super_df['Drug'].nunique() == 217, \
    f"ERROR: Expected 217 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 217 (216 drugs + no prescriptions).')

Unique persons: 127415
Unique drugs:   217

Top 10 most prescribed drugs:
Drug
no prescriptions    98066
atorvastatin        38542
lisinopril          35831
metformin           33774
amlodipine          28130
metoprolol          24981
albuterol           23188
omeprazole          22730
losartan            18670
gabapentin          18335
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
gentamicin          64
piroxicam           61
sulfamethoxazole    57
trimethoprim        57
ivermectin          29
Name: count, dtype: int64

Missing values per column:
Observation_ID             0
Person_ID                  0
Household_ID               0
Drug                       0
Age                     5261
Sex                        0
Family_income              0
Insurance_coverage         0
Race_ethnicity             0
Year                       0
Quantity               98066
Form                   98066
Strength               98146
Day_Supply            330224
dtype: int64

Missing value int

In [ ]:
# Define columns and fit LabelEncoder once on the full dataset.
# LabelEncoder is fitted before the CV loop so the drug->integer mapping
# is identical across all folds and for the final model.
feature_cols          = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity',
                         'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols      = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_demo_cols     = ['Age', 'Family_income']             
numeric_rx_cols       = ['Quantity', 'Strength', 'Day_Supply']  
target_col            = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes in encoder:', len(le.classes_))
print('Feature columns:          ', feature_cols)
print('Categorical cols:         ', categorical_cols)
print('Numeric demo cols:        ', numeric_demo_cols)
print('Numeric Rx cols:          ', numeric_rx_cols)
print('\nSample drug->integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'xgboost_super_label_encoder.joblib')
print('\nLabelEncoder saved to xgboost_super_label_encoder.joblib')

Unique classes in encoder: 217
Feature columns:           ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Categorical cols:          ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
Numeric demo cols:         ['Age', 'Family_income']
Numeric Rx cols:           ['Quantity', 'Strength', 'Day_Supply']

Sample drug->integer mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4

LabelEncoder saved to xgboost_super_label_encoder.joblib


In [ ]:
# set structural NaN's to -1 (this is necessary to differentiate between random
# missingness and the structural missingness of "no prescriptions")
mask = super_df['Drug'] == 'no prescriptions'
super_df.loc[mask, numeric_rx_cols] = -1
super_df.loc[mask, 'Form'] = '-1'

In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Applied within each fold to prevent data leakage.
# Consistent with all other base models.

def fit_age_medians(df):
    df = df.copy()

    income_bin_edges = {}
    income_bracket = pd.Series(index=df.index, dtype='float64')
    for year, group in df.groupby('Year'):
        try:
            bins, edges = pd.qcut(
                group['Family_income'], 4, labels=False,
                duplicates='drop', retbins=True
            )
            income_bin_edges[year] = edges
            income_bracket.loc[group.index] = bins + 1
        except ValueError:
            income_bin_edges[year] = None
    df['income_bracket'] = income_bracket

    hh_medians    = df.groupby(['Year', 'Household_ID'])['Age'].median()
    grp_medians   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].median()
    yr_medians    = df.groupby('Year')['Age'].median()
    global_median = df['Age'].median()

    return {
        'income_bin_edges': income_bin_edges,
        'hh_medians': hh_medians,
        'grp_medians': grp_medians,
        'yr_medians': yr_medians,
        'global_median': global_median,
        'available_years': sorted(yr_medians.index.unique().tolist()),
    }

def _nearest_year(year, available_years):
    if year in available_years:
        return year
    return min(available_years, key=lambda y: abs(y - year))

def _assign_income_bracket(df, income_bin_edges, available_years):
    income_bracket = pd.Series(index=df.index, dtype='float64')
    for year, group in df.groupby('Year'):
        effective_year = _nearest_year(year, available_years)
        edges = income_bin_edges.get(effective_year)
        if edges is not None:
            binned = pd.cut(group['Family_income'], bins=edges, labels=False, include_lowest=True)
            income_bracket.loc[group.index] = binned + 1
    return income_bracket

def apply_age_medians(df, medians):
    df = df.copy()
    available_years = medians['available_years']

    df['income_bracket'] = _assign_income_bracket(df, medians['income_bin_edges'], available_years)

    # Map every row's Year to its nearest training year for all lookups below.
    effective_year = df['Year'].apply(lambda y: _nearest_year(y, available_years))

    hh_idx    = pd.MultiIndex.from_arrays([effective_year, df['Household_ID']])
    hh_lookup = pd.Series(hh_idx.map(medians['hh_medians']), index=df.index)

    grp_idx    = pd.MultiIndex.from_arrays([effective_year, df['income_bracket'], df['Insurance_coverage']])
    grp_lookup = pd.Series(grp_idx.map(medians['grp_medians']), index=df.index)

    yr_lookup = effective_year.map(medians['yr_medians'])

    df['Age'] = df['Age'].fillna(hh_lookup)
    df['Age'] = df['Age'].fillna(grp_lookup)
    df['Age'] = df['Age'].fillna(yr_lookup)
    df['Age'] = df['Age'].fillna(medians['global_median'])
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def fit_strength_medians(df):
    mask = df['Drug'] != 'no prescriptions'
    yr_drug_medians = df.loc[mask].groupby(['Year', 'Drug'])['Strength'].median()
    drug_medians    = df.loc[mask].groupby('Drug')['Strength'].median()
    global_median   = df.loc[mask, 'Strength'].median()
    return {
        'yr_drug_medians': yr_drug_medians,
        'drug_medians': drug_medians,
        'global_median': global_median,
    }

def apply_strength_medians(df, medians):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'

    yr_drug_idx = pd.MultiIndex.from_frame(df.loc[mask, ['Year', 'Drug']])
    yr_drug_lookup = pd.Series(yr_drug_idx.map(medians['yr_drug_medians']), index=df.loc[mask].index)
    drug_lookup = df.loc[mask, 'Drug'].map(medians['drug_medians'])

    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(yr_drug_lookup)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(drug_lookup)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(medians['global_median'])
    return df

def fit_day_supply_medians(df):
    mask = df['Drug'] != 'no prescriptions'
    yr_drug_medians = df.loc[mask].groupby(['Year', 'Drug'])['Day_Supply'].median()
    drug_medians    = df.loc[mask].groupby('Drug')['Day_Supply'].median()
    global_median   = df.loc[mask, 'Day_Supply'].median()
    return {
        'yr_drug_medians': yr_drug_medians,
        'drug_medians': drug_medians,
        'global_median': global_median,
    }

def apply_day_supply_medians(df, medians):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'

    yr_drug_idx = pd.MultiIndex.from_frame(df.loc[mask, ['Year', 'Drug']])
    yr_drug_lookup = pd.Series(yr_drug_idx.map(medians['yr_drug_medians']), index=df.loc[mask].index)
    drug_lookup = df.loc[mask, 'Drug'].map(medians['drug_medians'])

    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(yr_drug_lookup)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(drug_lookup)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(medians['global_median'])
    return df

In [ ]:
# StratifiedGroupKFold 5-fold CV for XGBoost Super Dataset.
#
# For each fold:
#   1. Split by Person_ID groups, stratified by Drug
#   2. Impute Age, Strength, Day_Supply within each fold (fit on train, apply to val)
#   3. Set categorical dtypes for XGBoost native handling
#   4. Convert to DMatrix with enable_categorical=True
#   5. Train XGBoost multi:softmax classifier with early stopping
#   6. Predict and compute all required metrics

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

# XGBoost parameters.
XGB_DEVICE = 'cuda' if xgb.build_info().get('USE_CUDA', False) else 'cpu'
print(f'XGBoost device: {XGB_DEVICE}')

xgb_params = {
    'objective':        'multi:softprob',
    'num_class':        len(le.classes_),
    'eval_metric':      'mlogloss',
    'device':           XGB_DEVICE,
    'tree_method':      'hist',
    'max_depth':        6,
    'learning_rate':    0.1,
    'subsample':        0.8,
    'colsample_bytree': 1.0,
    'random_state':     42,
    'verbosity':        1,
}
N_ROUNDS       = 100
EARLY_STOPPING = 20

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    # Split on super_df so imputation functions have access to all columns
    train_fold = super_df.iloc[train_idx].copy()
    val_fold   = super_df.iloc[val_idx].copy()

    # Impute Age, Strength, Day_Supply within each fold.
    # Imputation is applied on train and val separately to prevent data leakage.
    age_medians = fit_age_medians(train_fold)
    train_fold = apply_age_medians(train_fold, age_medians)
    val_fold   = apply_age_medians(val_fold, age_medians)

    strength_medians = fit_strength_medians(train_fold)
    train_fold = apply_strength_medians(train_fold, strength_medians)
    val_fold   = apply_strength_medians(val_fold, strength_medians)

    day_supply_medians = fit_day_supply_medians(train_fold)
    train_fold = apply_day_supply_medians(train_fold, day_supply_medians)
    val_fold   = apply_day_supply_medians(val_fold, day_supply_medians)

    X_train_fold = train_fold[feature_cols].copy()
    X_val_fold   = val_fold[feature_cols].copy()
    y_train_fold = train_fold['Drug_encoded'].values
    y_val_fold   = val_fold['Drug_encoded'].values

    # Set categorical dtypes — XGBoost handles encoding and missing values internally.
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs — train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Convert to DMatrix with enable_categorical=True.
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold, enable_categorical=True)
    dval   = xgb.DMatrix(X_val_fold,   label=y_val_fold,   enable_categorical=True)

    # Train
    evals_result = {}
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        evals=[(dtrain, 'train'), (dval, 'val')],
        evals_result=evals_result,
        verbose_eval=10,
        early_stopping_rounds=EARLY_STOPPING,
    )
    print(f'Fold {fold_num} training complete. Best round: {model.best_iteration}')

    # Predict
    #y_pred_fold = model.predict(dval).astype(int)
    # get it out of integer mode (expected indices from softmax but now softprob)
    proba_fold = model.predict(dval)
    # Since dval is a DMatrix, this returns the (N, 217) matrix automatically.
    # To get the metrics (Accuracy, etc.), we take the argmax:
    y_pred_fold = np.argmax(proba_fold, axis=1)

    # Metrics
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    evaluator       = ClassificationMetric(y_val_fold, y_pred_fold)
    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')
    macro_f2        = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2        = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold':             fold_num,
        'accuracy':         acc,
        'cohen_kappa':      kappa,
        'mcc':              mcc,
        'macro_precision':  macro_precision,
        'micro_precision':  micro_precision,
        'macro_recall':     macro_recall,
        'micro_recall':     micro_recall,
        'macro_f1':         macro_f1,
        'micro_f1':         micro_f1,
        'macro_f2':         macro_f2,
        'micro_f2':         micro_f2,
    })

    # Per-drug recall for ensemble model selection
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:     {acc:.4f}')
    print(f'  Cohen Kappa:  {kappa:.4f}')
    print(f'  MCC:          {mcc:.4f}')
    print(f'  Macro Recall: {macro_recall:.4f}')
    print(f'  Micro Recall: {micro_recall:.4f}')
    print(f'  Macro F2:     {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

XGBoost device: cuda

FOLD 1/5
Before age: 15
After age: 15
Train size: 726,282 | Val size: 177,858
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.64993	val-merror:0.65756
[10]	train-merror:0.44693	val-merror:0.47825
[20]	train-merror:0.42627	val-merror:0.46700
[30]	train-merror:0.41460	val-merror:0.46257
[40]	train-merror:0.40469	val-merror:0.46045
[50]	train-merror:0.39586	val-merror:0.45936
[60]	train-merror:0.38827	val-merror:0.45801
[70]	train-merror:0.38090	val-merror:0.45607
[80]	train-merror:0.37358	val-merror:0.45659
[90]	train-merror:0.36564	val-merror:0.45599
[99]	train-merror:0.35858	val-merror:0.45575
Fold 1 training complete. Best round: 89
Fold 1 results:
  Accuracy:     0.5443
  Cohen Kappa:  0.5308
  MCC:          0.5325
  Macro Recall: 0.4536
  Micro Recall: 0.5443
  Macro F2:     0.4536

FOLD 2/5
Before age: 15
After age: 15
Train size: 723,019 | Val size: 181,121
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.65054	val-merror:0.65257
[10]	train-mer

In [ ]:
# Summarize CV results — mean and std per metric across all 5 folds.
# These numbers go directly into the paper.

results_df = pd.DataFrame(fold_results)

# Hard stop if not all 5 folds completed.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('Per-fold results:')
print(results_df.to_string(index=False))

print('\nMean ± Std across 5 folds:')
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:25s}: {mean:.4f} ± {std:.4f}')

results_df.to_csv('xgboost_super_cv_results.csv', index=False)
print('\nCV results saved to xgboost_super_cv_results.csv')
print('All 5 folds confirmed complete.')

Per-fold results:
 fold  accuracy  cohen_kappa      mcc  macro_precision  micro_precision  macro_recall  micro_recall  macro_f1  micro_f1  macro_f2  micro_f2
    1  0.544254     0.530762 0.532526         0.502758         0.544254      0.453573      0.544254  0.459698  0.544254  0.453644  0.544254
    2  0.547231     0.534380 0.536207         0.509366         0.547231      0.458230      0.547231  0.462398  0.547231  0.457140  0.547231
    3  0.541524     0.528140 0.529897         0.508012         0.541524      0.456222      0.541524  0.462657  0.541524  0.456292  0.541524
    4  0.545938     0.532950 0.534523         0.518447         0.545938      0.460132      0.545938  0.470161  0.545938  0.461599  0.545938
    5  0.541765     0.528495 0.530199         0.505832         0.541765      0.450956      0.541765  0.459928  0.541765  0.452272  0.541765

Mean ± Std across 5 folds:
  accuracy                 : 0.5441 ± 0.0025
  cohen_kappa              : 0.5309 ± 0.0027
  mcc                   

In [ ]:
# Average per-drug recall across all 5 folds.
# This CSV goes for ensemble construction.

per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_XGBoost_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_XGBoost_Super', ascending=False)

print('Top 20 drugs by mean recall (super dataset):')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall (super dataset):')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does XGBoost Super recall?
# Matches the format used across all other model notebooks for direct comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_XGBoost_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0:    {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_XGBoost_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_XGBoost_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('xgboost_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to xgboost_super_per_drug_recall.csv')

Top 20 drugs by mean recall (super dataset):
            Drug  Mean_Recall_XGBoost_Super
      colchicine                   1.000000
no prescriptions                   1.000000
       lactulose                   1.000000
      tiotropium                   0.998464
     latanoprost                   0.998299
      tamsulosin                   0.997384
     liraglutide                   0.995425
     linaclotide                   0.995152
       albuterol                   0.995146
     fluticasone                   0.989868
       clonidine                   0.989163
     dorzolamide                   0.988624
     bimatoprost                   0.981000
      azelastine                   0.980138
     alendronate                   0.976007
         aspirin                   0.973436
    moxifloxacin                   0.972324
       metformin                   0.968949
   nitroglycerin                   0.964170
   chlorhexidine                   0.955412

Bottom 20 drugs by mean recall

In [ ]:
# Train final XGBoost model on the full super integrated dataset (2014-2021).

# Impute Age, Strength, Day_Supply on full dataset before final training.
age_medians_final = fit_age_medians(super_df)
strength_medians_final = fit_strength_medians(super_df)
day_supply_medians_final = fit_day_supply_medians(super_df)

joblib.dump(age_medians_final, 'xgboost_super_age_medians.joblib')
joblib.dump(strength_medians_final, 'xgboost_super_strength_medians.joblib')
joblib.dump(day_supply_medians_final, 'xgboost_super_day_supply_medians.joblib')

data_final = apply_age_medians(super_df.copy(), age_medians_final)
fdata_final = apply_strength_medians(data_final, strength_medians_final)
data_final = apply_day_supply_medians(data_final, day_supply_medians_final)

X_final = data_final[feature_cols].copy()
y_final = data_final['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

# Shuffle to separate refill records after imputation.
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:       ', len(le.classes_))
print(f'Device:                   {XGB_DEVICE}')

# Quantity NaNs passed through as NaN — XGBoost learns missing direction natively.
dtrain_final = xgb.DMatrix(X_final, label=y_final, enable_categorical=True)

print('\nTraining final XGBoost model on super dataset...')
final_model = xgb.train(
    xgb_params,
    dtrain_final,
    num_boost_round=N_ROUNDS,
    verbose_eval=10,
)
print('Final model training complete.')

# Save final model in XGBoost binary format.
# .ubj is XGBoost's recommended binary format — smaller and faster than JSON.
# Can be reloaded with: model = xgb.Booster(); model.load_model('xgboost_super_final_model.ubj')
final_model.save_model('xgboost_super_final_model.ubj')
print('Model saved to xgboost_super_final_model.ubj')

Final model Family_income median: 38940.0
Final training data size: (904140, 9)
Number of classes:        217
Device:                   cuda

Training final XGBoost model on super dataset...
Final model training complete.
Model saved to xgboost_super_final_model.ubj
Train medians saved to xgboost_super_train_medians.json
